In [22]:

# Library imports
import pandas as pd
import numpy as np
import mlflow
from sklearn.model_selection import train_test_split
from pyspark.sql.functions import col, to_date, datediff, unix_timestamp, lead, when, year
from pyspark.sql.types import IntegerType
from pyspark.sql import Window
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
import requests
import json
from urllib.parse import quote
import logging
from datetime import datetime
import matplotlib.pyplot as plt

In [23]:
# Load the California housing dataset
housing = fetch_california_housing()
df_features = pd.DataFrame(housing.data, columns=housing.feature_names)
df_target = pd.DataFrame(housing.target, columns=housing.target_names)
df_combined = pd.concat([df_features, df_target], axis=1)  # add columns


In [24]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df):
    # Rename column 'MedInc' to 'MedianIncome'
    df = df.rename(columns={'MedInc': 'MedianIncome'})
    # Rename column 'AveRooms' to 'AvgRooms'
    df = df.rename(columns={'AveRooms': 'AvgRooms'})
    # Rename column 'AveBedrms' to 'AvgBedrms'
    df = df.rename(columns={'AveBedrms': 'AvgBedrms'})
    # Round down column 'AvgRooms'
    df[['AvgRooms']] = np.floor(df[['AvgRooms']])
    # Round column 'AveOccup' (Number of decimals: 1)
    df = df.round({'AveOccup': 1})
    # Round column 'AvgBedrms' (Number of decimals: 1)
    df = df.round({'AvgBedrms': 1})
    df['MedianIncome'] = df['MedianIncome'] * 10000
    df['MedHouseVal'] = df['MedHouseVal'] * 100000
    # Round up column 'MedianIncome'
    df[['MedianIncome']] = np.ceil(df[['MedianIncome']])
    return df

df_clean = clean_data(df_combined.copy())
df_clean.head()

,MedianIncome,HouseAge,AvgRooms,AvgBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,83252.0,41.0,6.0,1.0,322.0,2.6,37.88,-122.23,452600.0
1,83014.0,21.0,6.0,1.0,2401.0,2.1,37.86,-122.22,358500.0
2,72574.0,52.0,8.0,1.1,496.0,2.8,37.85,-122.24,352100.0
3,56431.0,52.0,5.0,1.1,558.0,2.5,37.85,-122.25,341300.0
4,38462.0,52.0,6.0,1.1,565.0,2.2,37.85,-122.25,342200.0


In [25]:
"""
Data Engineering.
"""

def Engineering_data(df):
    # Rename column 'MedInc' to 'MedianIncome'
    df = df.rename(columns={'MedInc': 'MedianIncome'})
    # Rename column 'AveRooms' to 'AvgRooms'
    df = df.rename(columns={'AveRooms': 'AvgRooms'})
    # Rename column 'AveBedrms' to 'AvgBedrms'
    df = df.rename(columns={'AveBedrms': 'AvgBedrms'})
    # Round down column 'AvgRooms'
    df[['AvgRooms']] = np.floor(df[['AvgRooms']])
    # Round column 'AveOccup' (Number of decimals: 1)
    df = df.round({'AveOccup': 1})
    # Round column 'AvgBedrms' (Number of decimals: 1)
    df = df.round({'AvgBedrms': 1})
    return df

df_engineered = Engineering_data(df_combined.copy())
df_engineered.head()

,MedianIncome,HouseAge,AvgRooms,AvgBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.0,1.0,322.0,2.6,37.88,-122.23,4.526
1,8.3014,21.0,6.0,1.0,2401.0,2.1,37.86,-122.22,3.585
2,7.2574,52.0,8.0,1.1,496.0,2.8,37.85,-122.24,3.521
3,5.6431,52.0,5.0,1.1,558.0,2.5,37.85,-122.25,3.413
4,3.8462,52.0,6.0,1.1,565.0,2.2,37.85,-122.25,3.422
